# Track A, step A4: fine-tune the DINOv2 embedding (GPU)
Switch the runtime to **GPU (T4)**: Runtime > Change runtime type. Run **A1 to A3 first** (this needs `img_pool.zip`, `pool.json`, `splits.json` on Drive). Status: `embed_ft.py` was only syntax-checked, never run on a GPU. Do the smoke test cell first.

In [ ]:
# 0) setup
GITHUB_USER = "YOUR_GITHUB_USERNAME"
import os, sys, json, random, numpy as np, pandas as pd
if not os.path.exists("/content/returnlens"):
    !git clone https://github.com/{GITHUB_USER}/returnlens.git /content/returnlens
%cd /content/returnlens
!git checkout track-a && git pull
sys.path.insert(0, "/content/returnlens/src")
from google.colab import drive; drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/returnlens_results"
!cd / && unzip -q -o "{OUT}/img_pool.zip"            # restores /content/img_amazon etc. (paths in pool.json point there)
POOL = json.load(open(f"{OUT}/pool.json")); SPLITS = json.load(open(f"{OUT}/splits.json"))
random.seed(0); np.random.seed(0)
print(len(POOL), "images;", {k: len(v) for k, v in SPLITS.items()}, "products per split")
!pip -q install transformers faiss-cpu opencv-python-headless imagehash
import torch; print(torch.cuda.get_device_name(0))

## Training sources (train products only)
Dictionary: product_id -> list of (source_id, PIL image). Products need at least 2 photos for hard negatives.

In [ ]:
from PIL import Image
import embed_ft
tr_prod = set(SPLITS["train"])
sources = {}
for p in POOL:
    if str(p["product_id"]) in tr_prod:
        sources.setdefault(p["product_id"], []).append((p["source_id"], Image.open(p["path"]).convert("RGB")))
sources = {k: v for k, v in sources.items() if len(v) >= 2}
print(len(sources), "train products,", sum(map(len, sources.values())), "images")
assert len(sources) >= 24, "need at least 24 products with 2+ photos; raise max_products in A1 or add Stanford Online Products"

## Smoke test: 50 steps. The loss must go DOWN. If it crashes, fix it before the long run.

In [ ]:
m = embed_ft.train(sources, steps=50, ckpt=f"/content/smoke.pt", n_products=12)

## Full run (about 3000 steps, roughly 1 to 1.5 hours on a T4). It saves a checkpoint every 100 steps to Drive; if Colab disconnects, re-run this cell with `resume=True`.

In [ ]:
CKPT = f"{OUT}/embedder.pt"
m = embed_ft.train(sources, steps=3000, ckpt=CKPT, resume=os.path.exists(CKPT))

## Recall@K: does fine-tuning help? (test products only)
Gallery = original test photos; query = an edited copy; success = the source photo is among the top K.

In [ ]:
import retrieval
test_pool = [p for p in POOL if str(p["product_id"]) in set(SPLITS["test"])]
print(len(test_pool), "test images")
res = {"pretrained DINOv2": retrieval.retrieval_eval(test_pool, retrieval.pretrained_dino()),
       "fine-tuned": retrieval.retrieval_eval(test_pool, retrieval.finetuned(CKPT))}
tab = pd.DataFrame(res).T.round(3); display(tab); tab.to_csv(f"{OUT}/recall_at_k.csv")

**Read this honestly.** If fine-tuning does not beat pre-trained DINOv2, report that: it is a valid result. With a small pool the gallery is tiny, so Recall@1 will already be high; use the largest test pool you have.